In [1]:
import pandas as pd
import numpy as np
import random
# import ast
from pyfaidx import Fasta
from torch.utils.data import Dataset, DataLoader
import torch

In [2]:
import sys
import os
sys.path.append(os.path.abspath("/home1/smaruj/pytorch_akita/"))

In [3]:
from model_v2_compatible import SeqNN

In [4]:
def one_hot_encode_sequence(sequence_obj):
    # Convert pyfaidx.Sequence object to string
    sequence = str(sequence_obj).upper()
    
    # Define the mapping from bases to integers
    base_to_int = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    valid_bases = list(base_to_int.keys())

    # Step 1: Convert sequence to integer encoding with random base for 'N'
    encoded_indices = []
    for base in sequence:
        if base in base_to_int:
            encoded_indices.append(base_to_int[base])
        else:
            random_base = random.choice(valid_bases)
            encoded_indices.append(base_to_int[random_base])

    # Step 2: One-hot encode the sequence
    encoded_sequence = np.array(encoded_indices)
    one_hot_encoded = np.zeros((4, len(encoded_sequence)), dtype=np.float32)
    one_hot_encoded[encoded_sequence, np.arange(len(encoded_sequence))] = 1

    return one_hot_encoded

In [5]:
class GenomicSequenceDataset(Dataset):
    def __init__(self, coord_df, genome_fasta, transform_fn=None):
        self.coords = coord_df  # DataFrame with chrom, start, end
        self.genome = genome_fasta
        self.transform_fn = transform_fn  # Optional function to modify sequence

    def __len__(self):
        return len(self.coords)

    def __getitem__(self, idx):
        TARGET_LEN = 1310720
        
        row = self.coords.iloc[idx]
        chrom, start, end = row["chrom"], row["start"], row["end"]
        seq = self.genome[chrom][start:end].seq.upper()
        
        # Fix sequence length if needed
        if len(seq) != TARGET_LEN:
            seq = seq[:TARGET_LEN].ljust(TARGET_LEN, 'N')  # pad with Ns if needed
        
        # Apply transformation, e.g. permute a window
        if self.transform_fn is not None:
            seq = self.transform_fn(seq, row)  # Pass row in case you want loc info
        
        one_hot = one_hot_encode_sequence(seq)  # shape: (4, L)
        return torch.from_numpy(one_hot.copy())

In [6]:
def set_diag(matrix, value, k):
    """Set diagonal `k` of a matrix to `value`."""
    rows, cols = matrix.shape
    for i in range(rows):
        if 0 <= i + k < cols:
            matrix[i, i + k] = value


def from_upper_triu_batch(batch_vectors, matrix_len=512, num_diags=2):
    """Convert a batch of upper-triangular vectors into symmetric matrices with np.nan on diagonals."""
    if isinstance(batch_vectors, torch.Tensor):
        batch_vectors = batch_vectors.detach().cpu().numpy()

    batch_size = batch_vectors.shape[0]
    matrices = np.zeros((batch_size, matrix_len, matrix_len), dtype=np.float32)

    triu_indices = np.triu_indices(matrix_len, num_diags)

    for i in range(batch_size):
        matrices[i][triu_indices] = batch_vectors[i]
        # Mirror to lower triangle
        matrices[i] = matrices[i] + matrices[i].T

        # Set diagonals to np.nan
        for k in range(-num_diags + 1, num_diags):
            set_diag(matrices[i], np.nan, k)

    return matrices  # shape: [B, 512, 512] 

In [7]:
BED_FILE = "/project/fudenber_735/tensorflow_models/akita/v2/data/mm10/sequences.bed"

In [8]:
df = pd.read_csv(BED_FILE, sep="\t", header=None, names=["chrom", "start", "end", "fold"])

In [9]:
FOLD = 0

In [10]:
df = df[df["fold"] == f"fold{FOLD}"].reset_index(drop=True)

In [ ]:
# df = df[:8]

In [11]:
fasta_file = "/project/fudenber_735/genomes/mm10/mm10.fa"
genome = Fasta(fasta_file)

In [12]:
orig_dataset = GenomicSequenceDataset(df, genome)

In [13]:
orig_loader = DataLoader(orig_dataset, batch_size=8, shuffle=False, drop_last=False)

In [14]:
# --- Load model ---
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

model = SeqNN()
model.load_state_dict(torch.load("/home1/smaruj/pytorch_akita/model_0_v2_finetuned_correctly.pt", map_location=device))
model.eval()

/tmp/SLURM_1349767/ipykernel_2103795/2795485947.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("/home1/smaruj/pytorch_akita/model_0_v2_

SeqNN(
  (stochastic_reverse_complement): StochasticReverseComplement()
  (stochastic_shift): StochasticShift()
  (conv_block_1): ConvBlock(
    (conv): Conv1d(4, 128, kernel_size=(15,), stride=(1,), padding=(7,), bias=False)
    (batch_norm): BatchNorm1d(128, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
    (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_tower): ConvTower(
    (conv_tower): Sequential(
      (0): ReLU()
      (1): Conv1d(128, 128, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
      (2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (4): ReLU()
      (5): Conv1d(128, 128, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
      (6): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (7): MaxPool1d(kernel_size=2, stride=2, paddi

In [15]:
orca_out_dir = "/scratch1/smaruj/orca_validation/fold0_orca_preds"

In [16]:
N = 256
diagonal_offset = 2

In [17]:
def vector_to_symmetric_matrix(vec, N):
    matrix = torch.zeros((N, N), dtype=vec.dtype)
    triu_indices = torch.triu_indices(N, N)
    matrix[triu_indices[0], triu_indices[1]] = vec
    matrix = matrix + matrix.T - torch.diag(torch.diag(matrix))
    return matrix

In [18]:
import matplotlib.pyplot as plt
import seaborn as sns

In [19]:
import torch.nn.functional as F

In [20]:
import scipy.stats

In [21]:
# Exclude diagonals: 0 and ±1
def get_upper_tri_mask(n, skip_diagonals=2):
    # Create mask with False on excluded diagonals, True elsewhere in upper triangle
    mask = np.triu(np.ones((n, n), dtype=bool), k=skip_diagonals)
    return mask

In [22]:
global_row_idx = 0  # tracks correct row in df
pearsonR = []

with torch.no_grad():
    for batch_idx, orig_batch in enumerate(orig_loader):
        
        print("index:", global_row_idx)
        
        orig_batch = orig_batch.squeeze(1)
        orig_preds = model(orig_batch.to(device)).cpu()
        og_maps = from_upper_triu_batch(orig_preds)
        
        # pooling 
        og_maps = torch.tensor(og_maps, dtype=torch.float32)
        og_maps = og_maps.unsqueeze(1)

        # Apply 2x2 average pooling
        og_maps_pooled = F.avg_pool2d(og_maps, kernel_size=2)  # -> (batch, 1, 256, 256)

        # Remove the channel dimension if not needed
        og_maps_pooled = og_maps_pooled.squeeze(1)  # -> (batch, 256, 256)
        og_maps_pooled = og_maps_pooled.numpy()
        
        current_batch_size = orig_preds.shape[0]
        
        # Save each prediction separately
        for i in range(current_batch_size):
            
            row = df.iloc[global_row_idx]
            chrom = row["chrom"]
            start = int(row["start"])
            end = int(row["end"])
            
            # plot for Akita
            akita_matrix = og_maps_pooled[i, :, :]
            
            # plt.figure(figsize=(6, 5))
            # sns.heatmap(akita_matrix, cmap="RdBu_r", square=True, cbar=True, vmin=-0.6, vmax=0.6)
            # plt.title(f"Akita Contact Map: {chrom}_{start}_{end}")
            # plt.xlabel("Bin")
            # plt.ylabel("Bin")
            # plt.tight_layout()
            # plt.show()

            filename = f"{chrom}_{start}_{end}_orca_pred.pt"
            file_path = f"{orca_out_dir}/{filename}"
            
            orca_vec = torch.load(file_path)
            orca_matrix = vector_to_symmetric_matrix(orca_vec, N).cpu().numpy()
            
            for i in range(-1, 2):
                diag_indices = np.diag_indices_from(orca_matrix)
                if i < 0:
                    orca_matrix[diag_indices[0][:i], diag_indices[1][-i:]] = np.nan
                elif i > 0:
                    orca_matrix[diag_indices[0][i:], diag_indices[1][:-i]] = np.nan
                else:
                    orca_matrix[diag_indices] = np.nan

            # plt.figure(figsize=(6, 5))
            # sns.heatmap(orca_matrix, cmap="RdBu_r", square=True, cbar=True, vmin=-0.6, vmax=0.6)
            # plt.title(f"ORCA Contact Map: {chrom}_{start}_{end}")
            # plt.xlabel("Bin")
            # plt.ylabel("Bin")
            # plt.tight_layout()
            # plt.show()
            
            mask = get_upper_tri_mask(akita_matrix.shape[0])
            
            akita_vals = akita_matrix[mask]
            orca_vals = orca_matrix[mask]

            # Remove NaNs (e.g., from diagonals)
            valid_mask = ~np.isnan(akita_vals) & ~np.isnan(orca_vals)
            akita_vals = akita_vals[valid_mask]
            orca_vals = orca_vals[valid_mask]
            
            # Calculate Pearson R
            if len(akita_vals) > 0:
                r, p_value = scipy.stats.pearsonr(akita_vals, orca_vals)
            else:
                r, p_value = np.nan, np.nan

            # print(f"Pearson R for {chrom}:{start}-{end}: {r:.4f}")
            
            pearsonR.append(r)
            
            global_row_idx += 1

index: 0


/tmp/SLURM_1349767/ipykernel_2103795/1484489743.py:48: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  orca_vec = torch.load(file_path)


index: 8
index: 16
index: 24
index: 32
index: 40
index: 48
index: 56
index: 64
index: 72
index: 80
index: 88
index: 96
index: 104
index: 112
index: 120
index: 128
index: 136
index: 144
index: 152
index: 160
index: 168
index: 176
index: 184
index: 192
index: 200
index: 208
index: 216
index: 224
index: 232
index: 240
index: 248
index: 256
index: 264
index: 272
index: 280
index: 288
index: 296
index: 304
index: 312
index: 320
index: 328
index: 336
index: 344
index: 352
index: 360
index: 368
index: 376
index: 384
index: 392
index: 400
index: 408
index: 416
index: 424
index: 432
index: 440
index: 448
index: 456
index: 464
index: 472
index: 480
index: 488
index: 496
index: 504
index: 512
index: 520
index: 528
index: 536
index: 544
index: 552
index: 560
index: 568
index: 576
index: 584
index: 592
index: 600
index: 608
index: 616
index: 624
index: 632
index: 640
index: 648
index: 656
index: 664
index: 672
index: 680
index: 688
index: 696
index: 704
index: 712
index: 720


In [24]:
average_r = sum(pearsonR) / len(pearsonR)
print(f"Average Pearson R: {average_r:.4f}")

Average Pearson R: 0.4581
